# Train NPE/MNPE From Config

Set `CONFIG_PATH` to the same JSON file you would pass to `scripts/train_npe_from_zarr.py`, then run the training cell. The notebook imports the training entry points from the script so it uses the same pipeline as the command line.


In [ ]:
import logging
import sys
from pathlib import Path

from IPython.display import JSON, display

CONFIG_PATH = Path("configs/model_inversion.baseline.train_mnpe_config.json")
LOG_LEVEL = "INFO"

raw_config_path = CONFIG_PATH.expanduser()
search_roots = []
if raw_config_path.is_absolute():
    search_roots.extend((raw_config_path.parent, *raw_config_path.parents))
cwd = Path.cwd().resolve()
search_roots.extend((cwd, *cwd.parents))

PROJECT_ROOT = None
for candidate in search_roots:
    if (candidate / "scripts" / "train_npe_from_zarr.py").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find project root containing scripts/train_npe_from_zarr.py")
if raw_config_path.is_absolute():
    CONFIG_PATH = raw_config_path.resolve()
else:
    CONFIG_PATH = (PROJECT_ROOT / raw_config_path).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from network_creation.model_inversion import train_inversion_from_config


## Config


In [ ]:
log_level = getattr(logging, LOG_LEVEL.upper())
logging.basicConfig(
    level=log_level,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logging.getLogger().setLevel(log_level)

display(JSON(filename=str(CONFIG_PATH)))


## Train


In [ ]:
result = train_inversion_from_config(CONFIG_PATH).to_dict()
display(JSON(result))
tensorboard_dir = result["tensorboard_dir"]
tensorboard_dir


## TensorBoard

Run this cell in notebook frontends that support the TensorBoard extension.


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir $tensorboard_dir
